# Day 5 — Fine-Tuned Models Evaluation & Comparative Analysis

Welcome to the **Day 5 Evaluation Notebook**! In this session, we will load both the **base** and **fine-tuned** versions of Qwen and Llama, run predictions on our unseen `test.json` dataset, and calculate standard NLP evaluation metrics:
- **BLEU Score** (n-gram precision)
- **ROUGE-1, ROUGE-2, ROUGE-L** (unigram, bigram, and longest common subsequence recall)
- **Response Length & Qualitative Quality**

At the end of the notebook, we will visualize the metrics with comparative bar charts and inspect side-by-side responses.

---  
## Step 1: Mount Google Drive & Install Required Packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes wandb trl rouge-score nltk pandas matplotlib

---  
## Step 2: Initialize Workspace & Sync Files

In [ ]:
import os

project_dir = "/content/Retail"
gdrive_dir = None
for candidate in ["/content/drive/MyDrive/Retail LLM", "/content/drive/MyDrive/Retail"]:
    if os.path.isdir(candidate):
        gdrive_dir = candidate
        break
if gdrive_dir is None:
    gdrive_dir = "/content/drive/MyDrive/Retail LLM"
    print(f"[!] Project folder not found on Drive. Defaulting target back to: {gdrive_dir}")
else:
    print(f"[+] Detected active Drive folder: {gdrive_dir}")

os.makedirs(project_dir, exist_ok=True)
print("[*] Syncing scripts and data from Google Drive...")
!rsync -av --progress "{gdrive_dir}/" /content/Retail/
print("[+] Workspace sync complete.")

---  
## Step 3: Write Latest Evaluation Script to Workspace

In [ ]:
evaluate_code = "import os\nimport argparse\nimport json\nimport torch\nimport random\nfrom datasets import load_dataset\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nfrom peft import PeftModel\nfrom rouge_score import rouge_scorer\nimport nltk\nfrom nltk.translate.bleu_score import sentence_bleu, SmoothingFunction\n\n# Download NLTK data if not present (handled quietly)\ntry:\n    nltk.data.find('tokenizers/punkt')\nexcept LookupError:\n    nltk.download('punkt', quiet=True)\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"Evaluate fine-tuned model against reference dataset.\")\n    parser.add_argument(\n        \"--model_id\",\n        type=str,\n        required=True,\n        help=\"Hugging Face base model identifier (e.g. Qwen/Qwen2.5-7B-Instruct or meta-llama/Meta-Llama-3-8B-Instruct)\"\n    )\n    parser.add_argument(\n        \"--adapter_dir\",\n        type=str,\n        default=None,\n        help=\"Path to the LoRA adapter directory. If None, evaluates the base model only.\"\n    )\n    parser.add_argument(\n        \"--test_file\",\n        type=str,\n        default=\"data/processed/test.json\",\n        help=\"Path to the test JSON file.\"\n    )\n    parser.add_argument(\n        \"--output_file\",\n        type=str,\n        required=True,\n        help=\"Path to save the evaluation results JSON file.\"\n    )\n    parser.add_argument(\n        \"--num_samples\",\n        type=int,\n        default=100,\n        help=\"Number of random samples to evaluate (default: 100).\"\n    )\n    parser.add_argument(\n        \"--seed\",\n        type=int,\n        default=42,\n        help=\"Random seed for reproducibility.\"\n    )\n    return parser.parse_args()\n\ndef main():\n    args = parse_args()\n    random.seed(args.seed)\n    \n    print(\"\\n==============================================\")\n    print(f\"[*] Base Model: {args.model_id}\")\n    print(f\"[*] Adapter Path: {args.adapter_dir}\")\n    print(f\"[*] Output Path: {args.output_file}\")\n    print(\"==============================================\\n\")\n    \n    # 1. Load Tokenizer\n    print(\"[*] Loading tokenizer...\")\n    tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n        \n    # 2. Load Model in 4-bit Quantization (to fit in T4 GPU VRAM)\n    print(\"[*] Loading base model in 4-bit quantization...\")\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_compute_dtype=torch.float16\n    )\n    \n    base_model = AutoModelForCausalLM.from_pretrained(\n        args.model_id,\n        quantization_config=bnb_config,\n        device_map=\"auto\",\n        trust_remote_code=True\n    )\n    \n    # Apply QLoRA dtypes casting sweep to avoid T4 gradient/precision issues\n    # Ensure all float16 model buffers are clean\n    for name, module in base_model.named_modules():\n        if \"norm\" in name or \"ln\" in name:\n            module.to(torch.float32)\n            \n    # 3. Load LoRA Adapter if provided\n    if args.adapter_dir:\n        print(f\"[*] Loading LoRA adapter from {args.adapter_dir}...\")\n        model = PeftModel.from_pretrained(base_model, args.adapter_dir)\n    else:\n        print(\"[*] No adapter provided. Evaluating raw base model.\")\n        model = base_model\n        \n    model.eval()\n    \n    # 4. Load Test Dataset\n    print(f\"[*] Loading test file: {args.test_file}...\")\n    if not os.path.exists(args.test_file):\n        raise FileNotFoundError(f\"Test file not found: {args.test_file}\")\n        \n    with open(args.test_file, \"r\", encoding=\"utf-8\") as f:\n        test_data = json.load(f)\n        \n    if len(test_data) > args.num_samples:\n        print(f\"[*] Sampling {args.num_samples} records from {len(test_data)} total test records.\")\n        # Ensure repeatable sampling using seeded random\n        test_samples = random.sample(test_data, args.num_samples)\n    else:\n        print(f\"[*] Using all {len(test_data)} test records.\")\n        test_samples = test_data\n        \n    # 5. Setup Scorers\n    rouge_scorer_inst = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)\n    smoothing = SmoothingFunction().method1\n    \n    results = []\n    total_r1, total_r2, total_rl, total_bleu = 0.0, 0.0, 0.0, 0.0\n    \n    # 6. Evaluation Generation Loop\n    print(\"\\n[*] Starting text generation and evaluation...\")\n    for idx, sample in enumerate(test_samples):\n        instruction = sample[\"instruction\"]\n        reference = sample[\"response\"]\n        \n        # Build prompt using SFT instruction-tuning prompt template\n        prompt = f\"Below is an instruction that describes a task. Write a response that appropriately completes the request.\\n\\n### Instruction:\\n{instruction}\\n\\n### Response:\\n\"\n        \n        inputs = tokenizer(prompt, return_tensors=\"pt\").to(\"cuda\")\n        \n        with torch.no_grad():\n            outputs = model.generate(\n                **inputs,\n                max_new_tokens=150,\n                temperature=0.7,\n                top_p=0.9,\n                do_sample=True,\n                pad_token_id=tokenizer.eos_token_id\n            )\n            \n        # Slice outputs to retrieve only the generated completion (ignoring prompt tokens)\n        prompt_len = inputs.input_ids.shape[1]\n        generation_tokens = outputs[0][prompt_len:]\n        prediction = tokenizer.decode(generation_tokens, skip_special_tokens=True).strip()\n        \n        # Compute ROUGE\n        rouge_scores = rouge_scorer_inst.score(reference, prediction)\n        r1 = rouge_scores['rouge1'].fmeasure\n        r2 = rouge_scores['rouge2'].fmeasure\n        rl = rouge_scores['rougeL'].fmeasure\n        \n        # Compute BLEU (word level)\n        ref_tokens = nltk.word_tokenize(reference.lower())\n        pred_tokens = nltk.word_tokenize(prediction.lower())\n        bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing)\n        \n        # Accumulate scores\n        total_r1 += r1\n        total_r2 += r2\n        total_rl += rl\n        total_bleu += bleu\n        \n        results.append({\n            \"instruction\": instruction,\n            \"reference\": reference,\n            \"prediction\": prediction,\n            \"metrics\": {\n                \"rouge1\": r1,\n                \"rouge2\": r2,\n                \"rougeL\": rl,\n                \"bleu\": bleu,\n                \"length\": len(prediction)\n            }\n        })\n        \n        if (idx + 1) % 10 == 0 or (idx + 1) == len(test_samples):\n            print(f\"    Processed {idx + 1}/{len(test_samples)} samples...\")\n            \n    # Calculate Summary Scores\n    num_evaluated = len(test_samples)\n    summary = {\n        \"mean_rouge1\": total_r1 / num_evaluated,\n        \"mean_rouge2\": total_r2 / num_evaluated,\n        \"mean_rougeL\": total_rl / num_evaluated,\n        \"mean_bleu\": total_bleu / num_evaluated\n    }\n    \n    output_data = {\n        \"model_id\": args.model_id,\n        \"adapter_dir\": args.adapter_dir,\n        \"summary\": summary,\n        \"results\": results\n    }\n    \n    # 7. Write Results\n    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)\n    with open(args.output_file, \"w\", encoding=\"utf-8\") as f:\n        json.dump(output_data, f, ensure_ascii=False, indent=2)\n        \n    print(\"\\n========================= SUMMARY =========================\")\n    print(f\"[+] ROUGE-1 F-Measure: {summary['mean_rouge1']:.4f}\")\n    print(f\"[+] ROUGE-2 F-Measure: {summary['mean_rouge2']:.4f}\")\n    print(f\"[+] ROUGE-L F-Measure: {summary['mean_rougeL']:.4f}\")\n    print(f\"[+] BLEU Score:        {summary['mean_bleu']:.4f}\")\n    print(\"===========================================================\\n\")\n    print(f\"[+] Detailed evaluation records saved to: {args.output_file}\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/evaluate.py", "w", encoding="utf-8") as f:
    f.write(evaluate_code)
print("[+] src/evaluate.py successfully written.")

---  
## Step 4: Evaluate Qwen Base vs. Fine-Tuned (Capped at 100 Samples)

In [ ]:
# 1. Run evaluation on Base Qwen Model
!python /content/Retail/src/evaluate.py \
    --model_id Qwen/Qwen2.5-7B-Instruct \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/qwen_base_results.json \
    --num_samples 100

In [ ]:
# 2. Run evaluation on Fine-Tuned Qwen Model
!python /content/Retail/src/evaluate.py \
    --model_id Qwen/Qwen2.5-7B-Instruct \
    --adapter_dir "{gdrive_dir}/models/qwen_v1" \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/qwen_finetuned_results.json \
    --num_samples 100

---  
## Step 5: Evaluate Llama Base vs. Fine-Tuned (Capped at 100 Samples)

In [ ]:
from google.colab import userdata
import os

if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        print("[+] Successfully loaded HF_TOKEN from Colab Secrets.")
    except Exception:
        print("[-] Warning: HF_TOKEN secret not found.")

# 1. Run evaluation on Base Llama Model
!python /content/Retail/src/evaluate.py \
    --model_id meta-llama/Meta-Llama-3-8B-Instruct \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/llama_base_results.json \
    --num_samples 100

In [ ]:
# 2. Run evaluation on Fine-Tuned Llama Model
!python /content/Retail/src/evaluate.py \
    --model_id meta-llama/Meta-Llama-3-8B-Instruct \
    --adapter_dir "{gdrive_dir}/models/llama_v1" \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/llama_finetuned_results.json \
    --num_samples 100

---  
## Step 6: Plot Comparative Results & Display Side-by-Side QA

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import os

eval_dir = "/content/Retail/models/evaluation"
models = {
    "Qwen Base": "qwen_base_results.json",
    "Qwen Fine-Tuned": "qwen_finetuned_results.json",
    "Llama Base": "llama_base_results.json",
    "Llama Fine-Tuned": "llama_finetuned_results.json"
}

metrics_summary = {}
for name, fname in models.items():
    fpath = os.path.join(eval_dir, fname)
    if os.path.exists(fpath):
        with open(fpath, "r") as f:
            data = json.load(f)
            metrics_summary[name] = data["summary"]
    else:
        print(f"[!] Warning: {fname} results file not found.")

if metrics_summary:
    df = pd.DataFrame(metrics_summary).T
    print("\n===================== OVERALL SCORES =====================")
    print(df.round(4))
    print("==========================================================\n")
    
    # Plot comparative chart
    fig, ax = plt.subplots(figsize=(10, 6))
    df[["mean_rouge1", "mean_rougeL", "mean_bleu"]].plot(kind="bar", ax=ax)
    ax.set_title("Base Model vs. Fine-Tuned Model Performance Comparison")
    ax.set_ylabel("Score (Higher is Better)")
    ax.set_xticklabels(df.index, rotation=15)
    ax.legend(["ROUGE-1", "ROUGE-L", "BLEU"])
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("[X] No evaluation metrics could be loaded.")

In [ ]:
# Show side-by-side responses for interactive validation
import pandas as pd
import random

qwen_ft_path = os.path.join(eval_dir, "qwen_finetuned_results.json")
qwen_base_path = os.path.join(eval_dir, "qwen_base_results.json")

if os.path.exists(qwen_ft_path) and os.path.exists(qwen_base_path):
    with open(qwen_ft_path) as f: ft_data = json.load(f)["results"]
    with open(qwen_base_path) as f: base_data = json.load(f)["results"]
    
    compare_list = []
    for i in range(min(5, len(ft_data))):  # Display 5 sample predictions
        compare_list.append({
            "Instruction": ft_data[i]["instruction"],
            "Reference Answer": ft_data[i]["reference"],
            "Base Model Response": base_data[i]["prediction"],
            "Fine-Tuned Response": ft_data[i]["prediction"]
        })
    
    df_compare = pd.DataFrame(compare_list)
    pd.set_option('display.max_colwidth', None)
    display(df_compare.style.set_properties(**{'text-align': 'left'}))
else:
    print("[!] Qwen result files not found to build comparison table.")

---  
## Step 7: Save Evaluation Results to Google Drive

In [ ]:
print(f"[*] Saving evaluation reports back to Drive folder: {gdrive_dir}...")
local_eval_path = "/content/Retail/models/evaluation"
drive_eval_path = os.path.join(gdrive_dir, "models", "evaluation")
os.makedirs(drive_eval_path, exist_ok=True)
!cp -v "{local_eval_path}/"*.json "{drive_eval_path}/"
print("[+] Backup complete.")